# Progress Review 1 — Kafka Weather Replay Producer

This notebook replays the downloaded ARCO-ERA5 JFK sample into Kafka topic `weather.raw`.

The source file is:

`data/sample/jfk_era5_2022_01_01_to_03.jsonl`

Each row is sent as one Kafka event.

In [1]:
import os
import json
import time
from pathlib import Path

from kafka import KafkaProducer

In [2]:
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_WEATHER_RAW")

required = {
    "KAFKA_BOOTSTRAP_SERVERS": KAFKA_BOOTSTRAP_SERVERS,
    "KAFKA_TOPIC_WEATHER_RAW": KAFKA_TOPIC,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise RuntimeError(f"Missing environment variables: {missing}")

SAMPLE_PATH = Path("data/sample/jfk_era5_2022_01_01_to_03.jsonl")

if not SAMPLE_PATH.exists():
    raise FileNotFoundError(
        f"Missing sample file: {SAMPLE_PATH}. "
        "Run Notebook 01 first."
    )

print("Kafka:", KAFKA_BOOTSTRAP_SERVERS)
print("Topic:", KAFKA_TOPIC)
print("Sample file:", SAMPLE_PATH)

Kafka: kafka:9092
Topic: weather.raw
Sample file: data/sample/jfk_era5_2022_01_01_to_03.jsonl


In [3]:
events = []

with SAMPLE_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            events.append(json.loads(line))

print("Loaded events:", len(events))
events[:2]

Loaded events: 72


[{'event_id': 'JFK_2022-01-01T00:00:00',
  'airport': 'JFK',
  'timestamp_utc': '2022-01-01 00:00:00',
  'latitude': 40.75,
  'longitude': 286.25,
  'temperature_k': 282.57415771484375,
  'temperature_c': 9.424163818359375,
  'wind_u_ms': -0.005778670310974121,
  'wind_v_ms': 1.8718559741973877,
  'wind_speed_ms': 1.871864914894104,
  'wind_speed_kts': 3.638605833053589,
  'precipitation_m': 5.76116144657135e-06,
  'precipitation_mm': 0.00576116144657135,
  'surface_pressure_pa': 101271.765625,
  'cape_j_kg': 12.700439453125},
 {'event_id': 'JFK_2022-01-01T01:00:00',
  'airport': 'JFK',
  'timestamp_utc': '2022-01-01 01:00:00',
  'latitude': 40.75,
  'longitude': 286.25,
  'temperature_k': 282.6150817871094,
  'temperature_c': 9.465087890625,
  'wind_u_ms': 0.25597989559173584,
  'wind_v_ms': 2.030057191848755,
  'wind_speed_ms': 2.0461323261260986,
  'wind_speed_kts': 3.977353811264038,
  'precipitation_m': 1.0564923286437988e-05,
  'precipitation_mm': 0.010564923286437988,
  'surface

In [4]:
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
    key_serializer=lambda v: v.encode("utf-8") if v else None,
)

print("Kafka producer created.")

Kafka producer created.


In [5]:
DELAY_SECONDS = 0.10

sent = 0

for event in events:
    key = event.get("airport", "UNKNOWN")

    producer.send(
        KAFKA_TOPIC,
        key=key,
        value=event,
    )

    producer.flush()
    sent += 1

    print(
        f"sent={sent:03d} "
        f"airport={event.get('airport')} "
        f"time={event.get('timestamp_utc')} "
        f"wind_kts={round(float(event.get('wind_speed_kts', 0.0)), 2)}"
    )

    time.sleep(DELAY_SECONDS)

producer.close()

print("Producer finished.")
print("Total events sent:", sent)

sent=001 airport=JFK time=2022-01-01 00:00:00 wind_kts=3.64
sent=002 airport=JFK time=2022-01-01 01:00:00 wind_kts=3.98
sent=003 airport=JFK time=2022-01-01 02:00:00 wind_kts=4.55
sent=004 airport=JFK time=2022-01-01 03:00:00 wind_kts=4.15
sent=005 airport=JFK time=2022-01-01 04:00:00 wind_kts=4.71
sent=006 airport=JFK time=2022-01-01 05:00:00 wind_kts=4.53
sent=007 airport=JFK time=2022-01-01 06:00:00 wind_kts=4.34
sent=008 airport=JFK time=2022-01-01 07:00:00 wind_kts=4.93
sent=009 airport=JFK time=2022-01-01 08:00:00 wind_kts=4.28
sent=010 airport=JFK time=2022-01-01 09:00:00 wind_kts=5.64
sent=011 airport=JFK time=2022-01-01 10:00:00 wind_kts=4.54
sent=012 airport=JFK time=2022-01-01 11:00:00 wind_kts=3.96
sent=013 airport=JFK time=2022-01-01 12:00:00 wind_kts=4.18
sent=014 airport=JFK time=2022-01-01 13:00:00 wind_kts=4.11
sent=015 airport=JFK time=2022-01-01 14:00:00 wind_kts=5.42
sent=016 airport=JFK time=2022-01-01 15:00:00 wind_kts=5.77
sent=017 airport=JFK time=2022-01-01 16: